# Plant Disease VQA — Google Colab training

Run cells manually in order. No benchmark launches automatically. Project code and temporary images live under /content; only durable artifacts are copied to Drive. Runtime setup and training cells require a Colab GPU.

## 1. Choose the project source

Set a public GitHub URL or upload a project ZIP/folder through the Files panel. Pin PROJECT_REF to a commit for reproducible runs. Do not clone into Drive.

In [ ]:
from pathlib import Path
import os

PROJECT_URL = "https://github.com/OWNER/Plant_disease_visual_question_answering.git"  # replace
PROJECT_REF = "main"  # preferably pin a commit SHA
UPLOAD_ZIP = ""  # optional: /content/plant-vqa-project.zip
PROJECT_DIR = Path("/content/plant_disease_visual_question_answering")
DATA_DIR = Path("/content/data/PlantVillageVQA")
OUTPUT_DIR = Path("/content/outputs")

if UPLOAD_ZIP:
    import zipfile
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOAD_ZIP) as zf:
        root = PROJECT_DIR.resolve()
        for member in zf.infolist():
            target = (PROJECT_DIR / member.filename).resolve()
            if not target.is_relative_to(root) or ".." in Path(member.filename).parts:
                raise ValueError(f"Unsafe project archive member: {member.filename}")
        zf.extractall(PROJECT_DIR)
elif not (PROJECT_DIR / "src/train.py").exists():
    if "OWNER" in PROJECT_URL:
        raise ValueError("Set PROJECT_URL to your GitHub repository or upload the project ZIP")
    !git clone --branch {PROJECT_REF} --depth 1 {PROJECT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Outputs:", OUTPUT_DIR)

## 2. Verify accelerator and install dependencies

The notebook never reinstalls torch. It installs the project requirements with torch/torchvision filtered out, then checks CUDA availability. Select a Colab runtime with NVIDIA T4 GPU.

In [ ]:
import subprocess, sys, importlib.util

for module in ("torch", "torchvision"):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f"Colab runtime is missing {module}; install a CUDA-enabled runtime first")
import torch
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime (Runtime > Change runtime type > T4 GPU)")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

requirements = (PROJECT_DIR / "requirements.txt").read_text().splitlines()
install = [line.strip() for line in requirements if line.strip() and not line.lstrip().startswith("#") and not line.split("==")[0].strip().lower() in {"torch", "torchvision"}]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *install])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)])
print("Installed project dependencies without reinstalling PyTorch.")

## 3. Mount Drive for durable storage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/PlantDiseaseVQA")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Durable outputs:", DRIVE_ROOT)

## 4. Download or locate dataset

Dataset is stored temporarily under /content. Download uses the repository CLI and Hugging Face revision discovery. To use an uploaded archive, extract it into DATA_DIR and verify it has one CSV with the six source columns. Never put the dataset in Drive for training.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
source_csv = list(DATA_DIR.rglob("*.csv"))
if not source_csv:
    subprocess.check_call([sys.executable, "-m", "src.download", "--data_dir", str(DATA_DIR)], cwd=PROJECT_DIR)
source_csv = list(DATA_DIR.rglob("*.csv"))
if len(source_csv) != 1:
    raise RuntimeError(f"Expected exactly one discovered annotation CSV, got {source_csv}; set an explicit --csv in later cells")
SOURCE_CSV = source_csv[0]
print("Annotation:", SOURCE_CSV)
print("Image files:", sum(1 for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}))

## 5. EDA and data/model tests

These checks do not train. The training gate also reruns data tests, validates all selected image decodes, and creates hash-bound evidence before it allows an optimizer to run.

In [ ]:
subprocess.check_call([sys.executable, "-m", "src.eda", "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV), "--output_dir", str(OUTPUT_DIR)], cwd=PROJECT_DIR)
subprocess.check_call([sys.executable, "-m", "src.verify_numbers", "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV), "--output", str(OUTPUT_DIR / "results/verification.json")], cwd=PROJECT_DIR)
subprocess.check_call([sys.executable, "-m", "pytest", "tests/test_data.py", "tests/test_dataset_pipeline.py", "tests/test_models.py", "tests/test_train_evaluate.py", "-v"], cwd=PROJECT_DIR)

## 6. Set the Colab resource profile

Base batch size is 16 with gradient accumulation 1. The T4 profile starts at batch 4 and accumulation 4 (effective batch size 16); lower peak VRAM, same nominal effective batch size. If CUDA OOM persists, use batch 2/accumulation 8, then batch 1/accumulation 16. Microbatch changes can alter BatchNorm/dropout/RNG behavior and optimizer update count if the effective batch changes; report the chosen settings. This model has no BatchNorm, but dropout sampling can still differ.

In [ ]:
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
NUM_WORKERS = 2

# Explicit override only: keep the original official test whenever possible.
TEST_POLICY = "subset"  # mini/class total budgets are smaller than all factoid test images

assert BATCH_SIZE * GRADIENT_ACCUMULATION == 16
print({"microbatch": BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
       "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION, "test_policy": TEST_POLICY})

## 7. Validate data for each planned run

Run this cell before each configuration. Validation runs selected-image decoding and pytest without skipped tests; no model is initialized in validation-only mode. It can take several minutes for full data.

In [ ]:
def validate_run(level, mode):
    report = OUTPUT_DIR / "results" / f"data_validation_{level}_subset.json"
    cmd = [sys.executable, "-m", "src.train", "--level", level, "--mode", mode,
           "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV),
           "--output_dir", str(OUTPUT_DIR), "--test-policy", TEST_POLICY,
           "--batch-size", str(BATCH_SIZE), "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION),
           "--num-workers", str(NUM_WORKERS), "--validate-only", "--validation-report", str(report)]
    subprocess.check_call(cmd, cwd=PROJECT_DIR)
    return report

print("Validation helper ready. No run has started.")

## 8. Train mini, class, ablations and full

Each code cell is an explicit user action. Cell outputs are saved locally first and copied to Drive after each run. This sequence is intentionally not executed on notebook open.

In [ ]:
def train_run(level, mode="full", *, resume=False):
    report = validate_run(level, mode)
    cmd = [sys.executable, "-m", "src.train", "--level", level, "--mode", mode,
           "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV),
           "--output_dir", str(OUTPUT_DIR), "--test-policy", TEST_POLICY,
           "--batch-size", str(BATCH_SIZE), "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION),
           "--num-workers", str(NUM_WORKERS), "--validation-report", str(report)]
    if resume:
        cmd.append("--resume")
    subprocess.check_call(cmd, cwd=PROJECT_DIR)
    backup_outputs()

def backup_outputs():
    import shutil
    for folder in ("models", "results", "figures"):
        source = OUTPUT_DIR / folder
        if source.exists():
            destination = DRIVE_ROOT / folder
            destination.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, destination, dirs_exist_ok=True)

print("Training helper ready. Call one run explicitly, for example train_run('mini', 'full').")

In [ ]:
train_run("mini", "full")

In [ ]:
train_run("class", "full")

In [ ]:
train_run("class", "frozen")

In [ ]:
train_run("class", "question_only")

In [ ]:
train_run("class", "image_only")

In [ ]:
train_run("full", "full")

## 9. Evaluate a selected checkpoint

Choose a concrete checkpoint path. Evaluation checks source, selected images, split and preprocessing fingerprints before inference. It evaluates the configured test view; subset policy is not the complete official test.

In [ ]:
CHECKPOINT = OUTPUT_DIR / "models/full_full.pt"  # edit after the desired run
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
subprocess.check_call([sys.executable, "-m", "src.evaluate", "--ckpt", str(CHECKPOINT),
    "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV), "--output_dir", str(OUTPUT_DIR / "results")], cwd=PROJECT_DIR)
backup_outputs()

## 10. Copy durable results to Drive

In [ ]:
backup_outputs()
print("Drive backup complete:", DRIVE_ROOT)
print("Checkpoints:", sorted(p.name for p in (DRIVE_ROOT / "models").glob("*.pt")))
print("Results:", len(list((DRIVE_ROOT / "results").glob("*"))))